<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/workflow-guides/workflow-03-analyze-fleet-in-ghanaian-eez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyze a fleet in Ghanaian EEZ

This guide provides detailed instructions to on how to use the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client) to **Monitor a Fleet (a group of vessels) of Tuna Longliners in [Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400) region for Compliance** using **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)**.

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Data Caveats](https://globalfishingwatch.org/our-apis/documentation#data-caveat), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be easily installed using pip:

In [1]:
# %pip install gfw-api-python-client

## Usage

Import and use `gfw-api-python-client` in your Python codes

In [2]:
import os

import gfwapiclient as gfw

In [3]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [4]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Introduction

**Use Case: Monitoring a Fleet of Tuna Longliners for Compliance**

Kwame is a fisheries compliance officer in Ghana, responsible for monitoring a fleet of tuna longliners operating within **[Ghanaian Exclusive Economic Zone (EEZ)](https://www.marineregions.org/gazetteer.php?p=details&id=8400)**. His goal is to:

1. Track apparent fishing effort for **longliners** over the last 12 months.

**API Used:**
️
1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** – Retrieve apparent fishing effort grouped by gear type in Ghanaian EEZ.

**Important:** In order to avoid any misinterpretation of **GFW data**, please refer to our official **data caveats** documentations:
- [Apparent fishing effort](https://globalfishingwatch.org/dataset-and-code-fishing-effort/) 
- [Exclusive economic zone boundaries definition](https://globalfishingwatch.org/our-apis/documentation#exclusive-economic-zone-boundaries-definition)

**Important Caveats:**

1. The [4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api) only supports **one active report per user at a time**.
2. **Sending multiple requests simultaneously** results in a **429 Too Many Requests** error.
3. If a report takes over **100 seconds** to generate, it may return a **524 Gateway Timeout** error.

## Step 0: Identify the Region of Interest (ROI) - Ghanaian EEZ

Before making API requests, Kwame must specify the geographic area for analysis using a **Region ID**:

**Options to Define the Region:**

1. **Using Region ID** - Each EEZ has a unique ID in the **[public-eez-areas](https://globalfishingwatch.org/our-apis/documentation#regions)** dataset.
2. **Custom Geometries** - Users can define a custom area using GeoJSON.
   
For **[Ghanaian EEZ, the region ID is 8400](https://www.marineregions.org/gazetteer.php?p=details&id=8400)** (public-eez-areas dataset).

## Step 1: Retrieve Fishing Effort in Ghanaian EEZ

Kwame **first queries** the **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** to get **fishing effort for all vessels**, grouping them by **gear type** in **[Ghanaian EEZ](https://www.marineregions.org/gazetteer.php?p=details&id=8400)**. Please [learn more about apparent fishing effort here](https://globalfishingwatch.org/our-apis/documentation#ais-apparent-fishing-effort) and [check its data caveats here](https://globalfishingwatch.org/our-apis/documentation#apparent-fishing-effort).

**Filters Used:**

1. **[Region ID](https://globalfishingwatch.org/our-apis/documentation#regions)** - 8400 Ghanaian EEZ
2. **[Date Range](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Last 12 Months
3. **[Grouped By](https://globalfishingwatch.org/our-apis/documentation#report-url-parameters-for-both-post-and-get-requests)** - Gear Type

In [5]:
step_1_report_result = await gfw_client.fourwings.create_fishing_effort_report(
    spatial_resolution="LOW",
    group_by="GEARTYPE",
    temporal_resolution="ENTIRE",
    start_date="2024-01-01",
    end_date="2025-01-01",
    spatial_aggregation=True,
    region={
        "dataset": "public-eez-areas",
        "id": "8400",
    },
)

In [6]:
step_1_report_df = step_1_report_result.df()

In [7]:
step_1_report_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   date                     3 non-null      str    
 1   detections               0 non-null      object 
 2   flag                     0 non-null      object 
 3   gear_type                3 non-null      str    
 4   hours                    3 non-null      float64
 5   vessel_ids               3 non-null      int64  
 6   vessel_id                0 non-null      object 
 7   vessel_type              0 non-null      object 
 8   entry_timestamp          0 non-null      object 
 9   exit_timestamp           0 non-null      object 
 10  first_transmission_date  0 non-null      object 
 11  last_transmission_date   0 non-null      object 
 12  imo                      0 non-null      object 
 13  mmsi                     0 non-null      object 
 14  call_sign                0 non-null      

In [8]:
step_1_report_df[["flag", "gear_type", "hours", "vessel_ids"]].head()

,flag,gear_type,hours,vessel_ids
0,None,fishing,2517.564722,9
1,None,tuna_purse_seines,3422.914722,20
2,None,trawlers,63320.477222,56


In [9]:
step_1_agg_report_df = (
    step_1_report_df.groupby(["gear_type"], as_index=False)
    .agg(hours=("hours", "sum"), vessel_ids=("vessel_ids", "sum"))
    .sort_values(by="hours", ascending=False)
)

In [10]:
step_1_agg_report_df

,gear_type,hours,vessel_ids
1,trawlers,63320.477222,56
2,tuna_purse_seines,3422.914722,20
0,fishing,2517.564722,9


### What We have Learned from Step 1

1. Kwame now has apparent fishing effort data for multiple gear types.

## Summary of API Flow

1. **[4Wings API](https://globalfishingwatch.org/our-apis/documentation#map-visualization-4wings-api)** - Identify fishing effort by **gear type** in Ghanaian EEZ.